# 12 Final Results

## Project objective
Can we build a credit default risk model that is accurate, explainable, and fair at **loan application time**?


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

ROOT = Path.cwd()
REPORTS = ROOT / 'reports'
MODELS = ROOT / 'models'
DATA_PATH = next((ROOT / 'data' / 'raw').glob('*'))
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)

def show_image_if_exists(path: Path, width: int = 900):
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing image: {path}')

raw_df = pd.read_excel(DATA_PATH)
comparison = pd.read_csv(REPORTS / 'model_validation' / 'clean_feature_model_comparison.csv')
temporal = pd.read_csv(REPORTS / 'model_validation' / 'temporal_split_comparison.csv')
leakage = load_json(REPORTS / 'leakage_audit' / 'leakage_audit_summary.json')
fairness = pd.read_csv(REPORTS / 'fairness_reports' / 'application_model' / 'xgboost_application_fairness_metrics.csv')
mitigation = load_json(REPORTS / 'fairness_reports' / 'application_model' / 'xgboost_application_bias_mitigation_summary.json')
cf = load_json(REPORTS / 'explainability_reports' / 'application_model' / 'xgboost_application_counterfactuals.json')
print('Dataset rows:', len(raw_df))
print('Original columns:', raw_df.shape[1])
display(comparison)


## Dataset summary

- **10,000 rows**
- **28 original columns**
- Target: `Default_Flag`
- Protected or sensitive attributes discussed in the project: `Gender`, `Age` / age grouping, and `Nationality` where usable

The project uses a case-study Dubai Arab Bank dataset and treats it as an academic responsible-AI exercise rather than a production banking scorecard.


## Leakage audit summary

The original full-feature XGBoost model looked almost perfect. That was treated as a red flag. The audit found no direct target leakage and no train/test overlap, but it did find that post-loan behavioral features created hindsight leakage for an application-time use case. The final model therefore excludes repayment-history and account-behavior fields.


In [ ]:
display(pd.DataFrame(leakage['top_single_feature_auc']).head())
print('Target shuffle ROC-AUC:', round(leakage['target_shuffle_test']['roc_auc'], 4))
print('Suspicious features flagged:')
for item in leakage['suspicious_features'][:8]:
    print('-', item)


## Model comparison

The clean application-time comparison is the main evidence for model selection. Behavioral monitoring and full diagnostic models are reported only as secondary references because they include post-loan signals.


In [ ]:
display(comparison[['model_name', 'feature_set', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']])
display(temporal[temporal['model_name'].str.contains('application')][['model_name', 'split_strategy', 'accuracy', 'roc_auc']])


## Final selected model

The final model is **`xgboost_application`** because it outperforms the logistic application model on ROC-AUC, remains stable under temporal validation, passes the target-shuffle sanity check, and supports explainability plus fairness analysis without relying on post-loan information.


## Explainability findings

Global SHAP explanations indicate that bureau score and loan-burden related variables are important risk drivers in the clean underwriting model. Local SHAP and LIME views provide applicant-level explanations for presentation and discussion.


In [ ]:
show_image_if_exists(REPORTS / 'explainability_reports' / 'application_model' / 'xgboost_application_shap_summary.png')
show_image_if_exists(REPORTS / 'explainability_reports' / 'application_model' / 'xgboost_application_shap_local.png')
show_image_if_exists(REPORTS / 'explainability_reports' / 'application_model' / 'xgboost_application_lime_local.png')


## Fairness findings

Fairness was evaluated on application decisions derived from the clean model. Gender was the default protected attribute because it was consistently available in the data. Demographic parity difference is small, disparate impact is close to 1, and mitigation illustrates the tradeoff between parity improvements and predictive power.


In [ ]:
display(fairness)
display(pd.DataFrame([
    {'method': 'baseline', **mitigation['baseline']['performance'], **mitigation['baseline']['fairness']},
    {'method': 'reweighing', **mitigation['reweighing']['performance'], **mitigation['reweighing']['fairness']},
    {'method': 'fairlearn_postprocessing', **mitigation['fairlearn_postprocessing']['performance'], **mitigation['fairlearn_postprocessing']['fairness']},
]))


## Counterfactual guidance

Counterfactual explanations turn a high predicted risk into actionable guidance. In the saved example, the main levers are better bureau score and lower loan burden, which is more useful to stakeholders than a probability alone.


In [ ]:
feature_names = cf['feature_names']
original = pd.Series(cf['test_data'][0][0][: len(feature_names)], index=feature_names)
first_cf = pd.Series(cf['cfs_list'][0][0][: len(feature_names)], index=feature_names)
changes = pd.DataFrame({'original': original, 'counterfactual': first_cf})
changes = changes[changes['original'].astype(str) != changes['counterfactual'].astype(str)]
display(changes)




## Final limitations

- The dataset appears to be a case-study or simulated banking dataset
- This is not a production-grade scorecard
- Causal fairness and policy analysis are out of scope
- External validation on real public datasets remains future work

## Final conclusion

The project demonstrates a responsible AI workflow for credit risk: **data validation -> leakage audit -> clean modeling -> explainability -> fairness analysis -> mitigation -> dashboard communication**.
